# Równanie Schrödingera - Prezentacja implementacji numerycznej
*Julia Zalewska, Oliwia Marut, Natalia Chmiel*

# Architektura pakietu
Prezentowane kody opierają się na stworzonym przez nas pakiecie w języku Julia. 
Główny plik modułu `SchrodingerSolver.jl` spina cały projekt w spójną całość.

In [ ]:
module SchrodingerSolver
    using LinearAlgebra
    include("fdm.jl")
    include("numerov.jl")
    export solve_FDM, solve_Numerov, E_analytic
end

## Metoda Różnic Skończonych (FDM) - Implementacja i Wykresy

In [ ]:
using LinearAlgebra
using SparseArrays
using Plots

Poniższa funkcja buduje macierz Hamiltona dla wektora potencjału `V` rozłożonego na siatce przestrzennej `x` i rozwiązuje problem własny.

In [ ]:
function solve_FDM(V::Vector{Float64}, x::Vector{Float64}, m::Float64, planck::Float64; n_states::Int=3)
    N = length(x)
    dx = x[2] - x[1]

    # Współczynnik przy drugiej pochodnej
    C = planck^2 / (2.0 * m * dx^2) 

    # Przekątna główna i poboczna
    main_diag = V .+ (2.0 * C) 
    under_diag = fill(-C, N - 1)

    # Utworzenie rzadkiej macierzy trójprzekątnej
    H = SymTridiagonal(main_diag, under_diag)
    
    # Rozwiązanie problemu własnego
    eigenvalues, eigenvectors = eigen(H)
    
    # Wybór n_states najniższych stanów
    E = eigenvalues[1:n_states]
    psi = eigenvectors[:, 1:n_states]
    
    # Normalizacja funkcji falowych w przestrzeni ciągłej
    psi = psi ./ sqrt(dx)
    
    return E, psi
end

### Testy i wizualizacje wyników dla różnych potencjałów